<a href="https://colab.research.google.com/github/prince127-web/GenAI/blob/main/LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U openai langgraph langchain-core

In [5]:
import os
from openai import OpenAI
from google.colab import userdata

# Get OpenAI API key from Colab Secrets
api_key = userdata.get("openai")

if not api_key:
    raise ValueError(
        "OpenAI API key not found. Add 'OpenAI_API_Key' "
        "to Colab Secrets and enable Notebook access."
    )

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class ChatbotState(TypedDict):
    user_message: str
    chatbot_response: str
    is_valid: bool

In [8]:
def chatbot_node(state: ChatbotState):

    message = state["user_message"]

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=[
            {
                "role": "system",
                "content": (
                    "You are a helpful Study Assistant chatbot. "
                    "Answer the user's questions clearly, accurately, "
                    "and in a student-friendly way."
                )
            },
            {
                "role": "user",
                "content": message
            }
        ]
    )

    answer = response.output_text

    return {
        "chatbot_response": answer
    }

In [9]:
def validation_node(state: ChatbotState):

    response = state["chatbot_response"]

    valid = len(response.strip()) > 10

    return {
        "is_valid": valid
    }

In [10]:
def chatbot_router(state: ChatbotState):

    if state["is_valid"]:
        return "valid"

    return "invalid"

In [12]:
# Build the LangGraph

builder = StateGraph(ChatbotState)

# Add OpenAI chatbot node
builder.add_node("chatbot", chatbot_node)

# Add validation node
builder.add_node("validator", validation_node)

# Start -> Chatbot
builder.add_edge(START, "chatbot")

# Chatbot -> Validator
builder.add_edge("chatbot", "validator")

# Conditional routing
builder.add_conditional_edges(
    "validator",
    chatbot_router,
    {
        "valid": END,
        "invalid": "chatbot"
    }
)

# Compile graph
app = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [13]:
# STEP 7 - Run the chatbot

user_question = "What is Artificial Intelligence?"

result = app.invoke({
    "user_message": user_question
})

print("========== STUDY ASSISTANT CHATBOT ==========")
print("User:", result["user_message"])
print("Bot:", result["chatbot_response"])
print("Valid Response:", result["is_valid"])

========== STUDY ASSISTANT CHATBOT ==========
User: What is Artificial Intelligence?
Bot: Artificial Intelligence (AI) is the field of creating computer systems that can perform tasks that usually require human intelligence.

These tasks include:

- Learning from data
- Understanding language
- Recognizing images and speech
- Solving problems
- Making decisions
- Predicting outcomes

Examples of AI include virtual assistants like Siri and Alexa, recommendation systems on Netflix or YouTube, self-driving technology, and chatbots.

In simple terms, **AI enables machines to learn, reason, and act in ways that resemble human intelligence**.
Valid Response: True
